# 🔎 Image Search — EDA

**SuperAI SS4 | Level 1 | Hackathon 1** · Python-first rebuild

โน้ตบุ๊กนี้มีไว้ *ดูข้อมูล* อย่างเดียว — pipeline จริงอยู่ใน `src/img_search/` และรันผ่าน `make`

คำถามที่อยากตอบให้ได้ก่อนเขียนโค้ดสักบรรทัด:

1. หน้าตาข้อมูลแต่ละก้อนเป็นยังไง (`queries/`, `train/`, `test/`)
2. `train/` ที่โจทย์แจกมาเอาไปทำอะไรได้ ทั้งที่ไม่มี label ตรงกับ 22 คลาสเลย
3. ทำไมการฮาร์ดโค้ด `cosine > 0.75` เพื่อตัดคลาส 22 ถึงเป็นวิธีที่เปราะ


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image

from img_search.config import load_config
from img_search import data as data_mod
from img_search.embed import get_embeddings
from img_search.utils import setup_logging

setup_logging()
cfg = load_config('../configs/default.yaml')
MODEL = 'clip-l14'   # เปลี่ยนเป็นตัวที่ benchmark บอกว่าดีที่สุดได้

queries = data_mod.queries_index(cfg)
train   = data_mod.train_index(cfg)
test    = data_mod.test_index(cfg)
len(queries), len(train), len(test)


## 1. ในกล่องมีอะไรบ้าง


In [ ]:
print(f'queries : {len(queries):>5} รูป  ({queries["cls"].nunique()} คลาส, คลาสละ 1 รูป)')
print(f'train   : {len(train):>5} รูป  ({train["folder"].nunique()} โฟลเดอร์/แบรนด์)')
print(f'test    : {len(test):>5} รูป  (ไม่มี label)')

sizes = train.groupby('folder').size()
print('\nรูปต่อโฟลเดอร์:', sizes.describe()[['min','25%','50%','75%','max']].to_dict())


In [ ]:
def grid(paths, titles, cols=11, size=1.7, suptitle=''):
    rows = int(np.ceil(len(paths) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(size*cols, (size+0.25)*rows))
    for ax, p, t in zip(np.atleast_1d(axes).ravel(), paths, titles):
        with Image.open(p) as im:
            ax.imshow(im.convert('RGB').resize((160, 160)))
        ax.set_title(t, fontsize=7); ax.axis('off')
    for ax in np.atleast_1d(axes).ravel()[len(paths):]: ax.axis('off')
    fig.suptitle(suptitle); plt.tight_layout(); plt.show()

grid(queries['path'], [f'class {c}' for c in queries['cls']],
     suptitle='queries/ — 22 logos we are asked about')


โลโก้ที่ถามเป็นแบรนด์ในไทยเป็นหลัก (Shell, Honda, MaxValu, Starbucks, มาม่า, PT, Emporium, APPMAN, …)
และเป็น **ภาพโลโก้ล้วน ๆ ที่ครอปมาแล้ว** ไม่ใช่รูปถ่ายในสภาพจริง — จำไว้ตอนเลือกวิธี preprocess


In [ ]:
rng = np.random.default_rng(0)
sample = test.sample(33, random_state=0)
grid(sample['path'], ['']*len(sample), suptitle='test/ — ภาพที่ต้องทำนาย')


test เป็นภาพชนิดเดียวกับ queries เป๊ะ ๆ (โลโก้ครอปมาแล้ว) — ประกอบด้วยโลโก้ 22 แบรนด์ที่ถาม
ปนกับแบรนด์อื่นอีกเพียบ (GUCCI, ASUS, acer, AIS, Kerry Express, …) ซึ่งต้องตอบว่าคลาส 22


## 2. `train/` เอาไปทำอะไรได้

โจทย์แจก `train/train/<แบรนด์>/*.jpg` มาให้ 173 โฟลเดอร์ — แต่ไม่มีโฟลเดอร์ไหนบอกว่าตรงกับ query ไหน
notebook รอบเดิมเลยไม่แตะมันเลย ลองดูก่อนว่าข้างในเป็นอะไร


In [ ]:
folders = sorted(train['folder'].unique())
first = [train[train['folder'] == f]['path'].iloc[0] for f in folders[:33]]
grid(first, folders[:33], suptitle='train/train/ — โฟลเดอร์ละ 1 ตัวอย่าง')


**ข้อสังเกตสำคัญ**: มันคือคลังโลโก้แบรนด์ *อื่น* (Gourmet Market, UNIQLO, KING POWER, adidas,
Häagen-Dazs, AIS Fibre, …) หน้าตาแบบเดียวกับ test เป๊ะ แค่ไม่ใช่ 22 แบรนด์ที่ถาม

นั่นทำให้มันมีค่ามาก 2 ทาง:

| ใช้ทำอะไร | ยังไง |
|---|---|
| **negative gallery** | ทุกรูปในนี้คือคำตอบ "คลาส 22" ที่เป็นตัวอย่างจริง ~2,300 รูป |
| **validation set** | สุ่ม 22 โฟลเดอร์มาเป็นคลาสปลอม แล้วจำลองโจทย์ทั้งดุ้นได้เลย (`src/img_search/proxy.py`) |

ข้อควรระวังข้อเดียว: ถ้ามีโฟลเดอร์ไหนบังเอิญเป็นแบรนด์เดียวกับ query จริง ๆ
การปล่อยให้อยู่ฝั่ง negative จะทำให้คลาสนั้นพังทั้งคลาส — `make map` มีไว้หาเคสพวกนี้


In [ ]:
# หาโฟลเดอร์ที่ใกล้เคียง query แต่ละอันที่สุด
q_emb = get_embeddings(cfg, MODEL, queries['path'].tolist())
t_emb = get_embeddings(cfg, MODEL, train['path'].tolist())
sim = q_emb @ t_emb.T

rows = []
for i, cls in enumerate(queries['cls']):
    best = pd.Series(sim[i], index=train['folder'].to_numpy()).groupby(level=0).max().nlargest(2)
    rows.append({'cls': cls, 'closest_folder': best.index[0], 'score': best.iloc[0],
                 'runner_up': best.index[1], 'runner_up_score': best.iloc[1]})
pd.DataFrame(rows).sort_values('score', ascending=False).head(10)


คะแนนสูง ๆ ไม่กี่อันเท่านั้นที่เป็นแบรนด์เดียวกันจริง ที่เหลือคือ "โลโก้หน้าตาคล้ายกัน" เฉย ๆ
— ต้องตรวจด้วยตาเสมอ (`uv run python scripts/visualize.py map`)


## 3. ทำไม threshold ตัวเดียวถึงเปราะ

วิธีเดิมคือ: เอา test เทียบกับโลโก้ 22 รูป เอาคะแนนสูงสุด ถ้าน้อยกว่า 0.75 ให้ตอบ 22
ลองพล็อตดูว่าเส้นแบ่งนั้นอยู่ตรงไหนของการกระจายจริง


In [ ]:
test_emb = get_embeddings(cfg, MODEL, test['path'].tolist())
best_q_test  = (test_emb @ q_emb.T).max(axis=1)   # test  -> โลโก้ที่ถาม
best_q_train = (t_emb    @ q_emb.T).max(axis=1)   # train -> โลโก้ที่ถาม (เกือบทั้งหมด = คลาส 22 จริง)

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0.3, 1.0, 60)
ax.hist(best_q_test,  bins=bins, alpha=.6, label='test (ไม่รู้คำตอบ)', density=True)
ax.hist(best_q_train, bins=bins, alpha=.6, label='train (รู้ว่าเป็นคลาส 22)', density=True)
ax.axvline(0.75, color='k', ls='--', label='threshold ที่ notebook เดิมใช้')
ax.set_xlabel(f'cosine สูงสุดเทียบกับโลโก้ 22 รูป ({MODEL})'); ax.legend(); plt.show()


สองการกระจายนี้ **ทับกันหนัก** ไม่ว่าจะลากเส้นตรงไหนก็ต้องเสียอย่างใดอย่างหนึ่ง
และตำแหน่งที่ดีที่สุดยังขึ้นกับโมเดลด้วย (สเกล cosine ของแต่ละ backbone ไม่เท่ากัน)
การฮาร์ดโค้ด 0.75 จึงเป็นการจูน hyperparameter ที่สำคัญที่สุดของงานนี้แบบมองไม่เห็นอะไรเลย


## 4. negative gallery ช่วยยังไง

แทนที่จะถามว่า *"คล้ายพอหรือยัง"* (ต้องมีเส้นสัมบูรณ์) ให้ถามว่า
*"ใกล้โลโก้ที่ถาม หรือใกล้โลโก้แบรนด์อื่นมากกว่ากัน"* — เป็นการเปรียบเทียบ ไม่ใช่การตัดสินด้วยเลขลอย ๆ


In [ ]:
best_neg_test = (test_emb @ t_emb.T).max(axis=1)   # test -> โลโก้แบรนด์อื่นใน train
margin = best_q_test - best_neg_test

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(best_neg_test, best_q_test, s=6, alpha=.4)
lim = [min(best_neg_test.min(), best_q_test.min()), 1.0]
axes[0].plot(lim, lim, 'k--', lw=1)
axes[0].set_xlabel('ใกล้แบรนด์อื่นแค่ไหน'); axes[0].set_ylabel('ใกล้โลโก้ที่ถามแค่ไหน')
axes[0].set_title('เหนือเส้น = น่าจะเป็น 1 ใน 22')

axes[1].hist(margin, bins=60)
axes[1].axvline(0, color='k', ls='--')
axes[1].set_xlabel('margin = (คะแนน query) − (คะแนน negative)')
axes[1].set_title(f'ตกฝั่งลบ {100*(margin < 0).mean():.0f}% ของ test')
plt.tight_layout(); plt.show()


การกระจายของ margin แยกเป็นสองก้อนชัดกว่าคะแนนดิบมาก และเส้นแบ่งที่ 0 มีความหมายในตัวเอง
ไม่ต้องจูน — ซึ่งเป็นเหตุผลที่ `match.reject` ตั้งค่าเริ่มต้นเป็น `gallery+threshold`

---

ขั้นต่อไปดูที่ [`README.md`](../README.md): `make map` → `make benchmark` → `make predict`
